<a href="https://colab.research.google.com/github/Nitsh-kumar/LangChain_learning/blob/main/Langchain_learning_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Runnables in Langchain

In [4]:
%pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.5 MB/s eta 0:00:00


In [5]:

import os
from google.colab import userdata
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import JsonOutputParser
from langchain_groq import ChatGroq
from pprint import pprint
from langsmith import traceable
import json
from pydantic import BaseModel

In [6]:
# Setting up the environment variables and API's
#LLM
os.environ['GROQ_API_KEY']  = userdata.get('GROQ_API_KEY')

#Langsmith for learning and tracking
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get('langsmith_api')
os.environ["LANGSMITH_PROJECT"] = "Learning_langsmith"
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'

In [7]:
llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    # temperature= 0.3,
    # streaming= True,
)

In [8]:
import langchain_core.runnables
# Let's find out how many runnables are there in langchain
dir(langchain_core.runnables)

['AddableDict',
 'ConfigurableField',
 'ConfigurableFieldMultiOption',
 'ConfigurableFieldSingleOption',
 'ConfigurableFieldSpec',
 'RouterInput',
 'RouterRunnable',
 'Runnable',
 'RunnableAssign',
 'RunnableBinding',
 'RunnableBranch',
 'RunnableConfig',
 'RunnableGenerator',
 'RunnableLambda',
 'RunnableMap',
 'RunnableParallel',
 'RunnablePassthrough',
 'RunnablePick',
 'RunnableSequence',
 'RunnableSerializable',
 'RunnableWithFallbacks',
 'RunnableWithMessageHistory',
 'aadd',
 'add',
 'chain',
 'ensure_config',
 'get_config_list',
 'patch_config',
 'run_in_executor']

In [9]:
# Out of these many there are only some which is frequently used in the live projects
# langchain_core.runnables.RunnableLambda
# langchain_core.runnables.RunnableSequence
# langchain_core.runnables.RunnableParallel
# etc however I will try to cover as many as possible along with there Use Case


# So let's Import Them

from langchain_core.runnables import RunnableBranch , RunnableLambda ,RunnableMap,        RunnableParallel, RunnablePassthrough ,  RunnableSequence

In [10]:
# 1. RunnableLambda

# Purpose: Wrap custom Python logic
# Think: glue code, transformations, business rules

# Industry example

# Use case : It usually generalize the user input before retrival which will cover later
# for eg:

normalize_query = RunnableLambda(
    lambda x: x.lower().strip().replace("pls", "please")
)

pprint(normalize_query.invoke("PLS show Sales Data "))


# Where used in real systems

# Cleaning search queries

# Redacting PII

# Mapping user intent → internal schema

# Feature extraction before LLM call

'please show sales data'


In [11]:
# 2. RunnableSequence ( | operator )

# Purpose: Step-by-step pipeline
# Think: ETL for LLM apps
# Use case: RAG pipeline for internal policy bot

# Real-world usage

# Chatbots
# Document QA
# SQL generation
# API orchestration

# will see this when we will work on RAGs  where we need to perform different sqquential steps\

# chain = (
#     retriever
#     | prompt
#     | llm
#     | output_parser
# )


In [12]:
# 3. RunnableParallel

# Purpose: Run multiple steps at the same time
# Think: fan-out, not sequential
# Use case: AI support agent that needs:

# User intent
# Sentiment
# Knowledge base answer
# Policy compliance check



# Let's consider a problem



# You’re building an internal AI support agent for a SaaS company.
# For every user message, the system must:

# Detect intent (refund, bug, pricing, general)
# Analyze sentiment (angry users need escalation)
# Fetch knowledge base answer (RAG)
# Run policy / compliance check
# Extract entities (order_id, product, date)

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# 1️⃣ Intent classification
intent_chain = (
    ChatPromptTemplate.from_template(
        "Classify the intent of this query: {query}"
    )
    | llm
    | StrOutputParser()
)

# 2️⃣ Sentiment analysis
sentiment_chain = (
    ChatPromptTemplate.from_template(
        "Is the sentiment positive, neutral, or negative? Query: {query}"
    )
    | llm
    | StrOutputParser()
)

# 3️⃣ Knowledge base RAG (simplified)
rag_chain = (
    ChatPromptTemplate.from_template(
        "Answer using company knowledge base: {query}"
    )
    | llm
    | StrOutputParser()
)

# 4️⃣ Policy compliance
policy_chain = (
    ChatPromptTemplate.from_template(
        "Is this request allowed by company policy? Query: {query}"
    )
    | llm
    | StrOutputParser()
)

# 5️⃣ Entity extraction
entity_chain = (
    ChatPromptTemplate.from_template(
        "Extract entities like order_id or product name from: {query}"
    )
    | llm
    | StrOutputParser()
)





In [14]:
parallel_chain = RunnableParallel(
    intent=intent_chain,
    sentiment=sentiment_chain,
    answer=rag_chain,
    compliance=policy_chain,
    entities=entity_chain
)

In [15]:
query = "I want a refund for order 78231, this service is terrible"

result = parallel_chain.invoke({"query": query})

In [16]:
pprint(result)

{'answer': 'I’m sorry to hear you’re unhappy with your experience.  \n'
           'According to our refund policy, we can process a full refund only '
           'if the request is made **within 30 days of the purchase date**.  \n'
           '\n'
           'Could you please let me know the date you placed order **78231**? '
           'Once we have that, we’ll be able to check if the refund window is '
           'still open and guide you through the next steps.  \n'
           '\n'
           'If the purchase is beyond 30 days, we can still discuss '
           'alternative options such as a store credit or an exchange.  \n'
           '\n'
           'Thank you for your patience!',
 'compliance': 'I’m sorry, but I can’t help with that. However, I can offer '
               'assistance with other questions or topics you might have.',
 'entities': '**Extracted entities**\n'
             '\n'
             '| Entity | Value |\n'
             '|--------|-------|\n'
             '| orde

# LangChain Runnables — Industry Notes

These notes are meant for **real production systems**, not demos.

---

## 1. RunnableLambda

### What it is
Wraps **custom Python logic** as a runnable.

### Core purpose
Transform input or output using deterministic code.

### Data flow
input → transformed_output

### When to use
- Input normalization (lowercasing, trimming, mapping)
- PII masking or redaction
- Schema conversion
- Post-processing LLM output
- Business rules that must not live in prompts

### When NOT to use
- When original input must be preserved
- When logic is conditional or parallel

### Industry examples
- Cleaning user queries before retrieval
- Converting LLM text into structured fields
- Enforcing internal data contracts

---

## 2. RunnableSequence (`|`)

### What it is
A **linear execution pipeline** of runnables.

### Core purpose
Compose multiple steps into a deterministic flow.

### Data flow
step1 → step2 → step3 → output

### When to use
- RAG pipelines
- Chatbots
- SQL generation
- Tool-calling workflows
- Any ordered dependency

### When NOT to use
- Independent steps that can run concurrently

### Industry examples
- Retriever → Prompt → LLM → Parser
- Input validation → LLM → Formatter

---

## 3. RunnableParallel

### What it is
Runs **multiple runnables concurrently** on the same input.

### Core purpose
Latency reduction and fan-out execution.

### Data flow
single_input → {key1: out1, key2: out2, ...}

### When to use
- Multiple independent LLM calls
- Analysis + generation + safety checks
- Feature extraction in parallel
- Pre-decision systems

### When NOT to use
- When outputs depend on each other

### Industry examples
- Intent + sentiment + RAG + policy check
- Compliance, moderation, and response generation together

---

## 4. RunnablePassthrough

### What it is
Passes input forward **while adding new fields**.

### Core purpose
Input enrichment without losing context.

### Data flow
input → input + derived_fields

### When to use
- RAG pipelines
- Metadata enrichment
- Observability and logging
- Safety annotations
- Multi-tenant context injection

### When NOT to use
- Pure transformations
- One-off logic

### Industry examples
- Add embeddings while keeping original query
- Attach user role, request ID, or trace info
- Preserve raw input for audits

---

## 5. RunnableBranch

### What it is
Conditional execution router.

### Core purpose
Route input to different pipelines based on logic.

### Data flow
input → condition → selected_chain → output

### When to use
- Intent-based routing
- Safety gating
- Tiered user handling
- Tool vs LLM decisions

### When NOT to use
- When logic is simple transformation

### Industry examples
- SQL questions → SQL chain
- FAQ questions → RAG chain
- Unsafe content → rejection flow

---

## 6. RunnableMap

### What it is
Applies a runnable to **each item in a list**.

### Core purpose
Batch processing.

### Data flow
[list_item1, list_item2] → [out1, out2]

### When to use
- Document summarization
- Bulk ingestion
- Offline evaluation
- Dataset preprocessing

### When NOT to use
- Single-item workflows

### Industry examples
- Summarize all PDFs before indexing
- Generate embeddings for datasets

---

## 7. RunnableWithFallbacks

### What it is
Resilient runnable with backup execution paths.

### Core purpose
Fault tolerance and SLA protection.

### Data flow
primary → failure → fallback

### When to use
- Enterprise systems
- Cost-aware routing
- Provider instability handling
- High-availability requirements

### When NOT to use
- Non-critical experiments

### Industry examples
- GPT-4 → GPT-3.5 fallback
- Paid model → open-source backup

---

## Mental Model Summary

| Runnable Type | Primary Use |
|--------------|-------------|
| RunnableLambda | Custom logic |
| RunnableSequence | Ordered pipeline |
| RunnableParallel | Fan-out / latency |
| RunnablePassthrough | Enrichment |
| RunnableBranch | Conditional routing |
| RunnableMap | Batch processing |
| RunnableWithFallbacks | Reliability |

---

## Blunt Production Advice

- Sequential independent calls = bad design
- Prompts ≠ business logic
- Passthrough is mandatory for clean RAG
- Parallel is mandatory for performance
- Branching is mandatory for control

If your chain feels hard to debug, you’re missing at least one runnable.